In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
from langchain.tools import tool
from tavily import TavilyClient 
from typing import Dict , Any 

tavily_client = TavilyClient()

@tool 
def web_search(query : str) -> Dict[str , Any]: 
    """Web search for the recipes can be made with the items found """
    return tavily_client.search(query)





In [21]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
model = init_chat_model(model="gemini-3.5-flash" , model_provider="google_genai")


system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

agent = create_agent(model , system_prompt=system_prompt , tools=[web_search] , checkpointer=InMemorySaver())



In [8]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept=".png" , multiple=False )

display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [13]:
import base64

uploaded_file = uploader.value[0]
content_mv = uploaded_file["content"]
img_bytes  = bytes(content_mv)
img_b64   = base64.b64encode(img_bytes).decode("utf-8")



In [ ]:
from langchain.messages import HumanMessage

from pprint import pprint

multimodal_question = HumanMessage(content=[
    {
        "type":"text" , "text" : "Tell me what can make to eat from these leftover items , suggest something good to eat"
    } , 
    {
        "type" : "image" , "base64" : img_b64 , "mime_type" : "image/png"
    }
])
config = {"configurable" : {"thread_id" : "1"}}

response = agent.invoke(
    {
        "messages": [multimodal_question] ,  
        
    } , config 
)

pprint(response)


{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'Tell me what can make to eat from these leftover items , suggest something good to eat'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='e59a9aa6-1429-4a56-b2df-ab5f9f67365b'),
              AIMessage(content=[{'type': 'text', 'text': 'As a chef, looking at these ingredients excites me. You have the ultimate toolkit for creating **Shatteringly Crispy Twice-Fried Chicken**. \n\nThe combination of **all-purpose flour, cornstarch, and baking powder** is a professional secret. The cornstarch inhibits gluten formation (preventing chewiness), while the baking powder releases tiny carbon dioxide bubbles when hit with hot oil, creating an ultra-light, aerated, glass-like crunchy crust. Pair that with a dry brine of **salt and sugar** to lock in moisture, and you have a world-class dish.\n\nHere is how we elevate these simple ingredients into a Michelin-level crispy chic